In [1]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
import pymysql

In [2]:
DB_USER = "root"
DB_PASSWORD = 1234
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "olist_db"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [3]:
leads_df = pd.read_sql(
    "SELECT * FROM marketing_qualified_leads",
    engine
)

deals_df = pd.read_sql(
    "SELECT * FROM closed_deals",
    engine
)

In [4]:
print(leads_df.shape)
print()
print(deals_df.shape)

(8000, 4)

(842, 14)


In [5]:
funnel_df = leads_df.merge(
    deals_df,
    on="mql_id",
    how="left",
    suffixes=("_lead", "_deal")
)

In [6]:
funnel_df["is_converted"] = np.where(
    funnel_df["won_date"].notna(), 1, 0
)

In [7]:
funnel_df["is_converted"].value_counts()  #0 means people closed the acc

is_converted
0    7158
1     842
Name: count, dtype: int64

In [8]:
categorical_cols = [
    "origin",
    "business_segment",
    "lead_type",
    "lead_behaviour_profile",
    "business_type"
]

for col in categorical_cols:
    if col in funnel_df.columns:
        funnel_df[col] = funnel_df[col].fillna("Unknown")

In [9]:
funnel_df["lead_month"] = (
    funnel_df["first_contact_date"]
    .dt.to_period("M")
    .astype(str)
)

funnel_df["conversion_month"] = (
    funnel_df["won_date"]
    .dt.to_period("M")
    .astype(str)
)

In [10]:
funnel_df.to_sql(
    name="funnel_base",
    con=engine,
    if_exists="replace",
    index=False
)

8000

In [13]:
pd.read_sql(
    "SELECT COUNT(*) AS row_no FROM funnel_base",
    engine
)

,row_no
0,8000
